# Analise Exploratoria de Dados - Gelato Magico

Projeto de Machine Learning para previsao de vendas de sorvete baseado em temperatura e outros fatores.

**Autor:** Gabriel Demetrios Lafis

**Bootcamp:** Microsoft Certification Challenge #5 - DP-100 (DIO)

In [ ]:
# Setup e importacoes
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Adicionar raiz do projeto ao sys.path para importar modulos locais
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

%matplotlib inline

# Configuracao de estilo dos graficos
sns.set_style("whitegrid")
sns.set_palette("viridis")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12

print("Setup concluido com sucesso!")
print(f"Raiz do projeto: {PROJECT_ROOT}")

## 1. Carregamento dos Dados

Vamos carregar os dois datasets disponiveis:
- **ice_cream_sales_original.csv**: Dataset original com dados simples de temperatura e receita.
- **gelato_magico_vendas.csv**: Dataset enriquecido com features adicionais (dia da semana, feriados, estacao).

In [ ]:
from src.data_preparation import load_data

# Caminhos dos datasets
INPUTS_DIR = PROJECT_ROOT / "inputs"

# Carregar dataset original
df_original = load_data(INPUTS_DIR / "ice_cream_sales_original.csv")
print("\n--- Dataset Original ---")
display(df_original.head())
print()
df_original.info()
print("\nEstatisticas descritivas:")
display(df_original.describe())

In [ ]:
# Carregar dataset Gelato Magico (enriquecido)
df_gelato = load_data(INPUTS_DIR / "gelato_magico_vendas.csv")
print("\n--- Dataset Gelato Magico ---")
display(df_gelato.head(10))
print()
df_gelato.info()
print("\nEstatisticas descritivas:")
display(df_gelato.describe())

## 2. Resumo Estatistico

Analise detalhada das estatisticas descritivas, tipos de dados, valores ausentes e correlacoes.

In [ ]:
# Estatisticas descritivas completas do dataset principal
print("=" * 60)
print("  RESUMO ESTATISTICO - GELATO MAGICO")
print("=" * 60)

print(f"\nFormato do DataFrame: {df_gelato.shape[0]} linhas x {df_gelato.shape[1]} colunas")

print("\n--- Tipos de Dados ---")
display(df_gelato.dtypes)

print("\n--- Estatisticas Descritivas ---")
display(df_gelato.describe())

print("\n--- Valores Ausentes ---")
missing = df_gelato.isnull().sum()
missing_pct = (df_gelato.isnull().sum() / len(df_gelato)) * 100
missing_df = pd.DataFrame({"Ausentes": missing, "Percentual (%)": missing_pct})
display(missing_df)

print("\n--- Matriz de Correlacao ---")
numeric_cols = df_gelato.select_dtypes(include=[np.number])
corr_matrix = numeric_cols.corr()
display(corr_matrix)

## 3. Analise Univariada

Distribuicao individual das variaveis principais: temperatura e vendas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma da temperatura
sns.histplot(
    df_gelato["temperatura"], bins=30, kde=True,
    color="#2196F3", ax=axes[0]
)
axes[0].set_title("Distribuicao da Temperatura", fontsize=16, fontweight="bold")
axes[0].set_xlabel("Temperatura (°C)", fontsize=12)
axes[0].set_ylabel("Frequencia", fontsize=12)
axes[0].axvline(
    df_gelato["temperatura"].mean(), color="red",
    linestyle="--", linewidth=2, label=f'Media: {df_gelato["temperatura"].mean():.1f}°C'
)
axes[0].legend(fontsize=11)

# Histograma das vendas
sns.histplot(
    df_gelato["vendas"], bins=30, kde=True,
    color="#4CAF50", ax=axes[1]
)
axes[1].set_title("Distribuicao das Vendas", fontsize=16, fontweight="bold")
axes[1].set_xlabel("Vendas (R$)", fontsize=12)
axes[1].set_ylabel("Frequencia", fontsize=12)
axes[1].axvline(
    df_gelato["vendas"].mean(), color="red",
    linestyle="--", linewidth=2, label=f'Media: R$ {df_gelato["vendas"].mean():.2f}'
)
axes[1].legend(fontsize=11)

plt.suptitle("Analise Univariada - Variaveis Principais", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"Temperatura - Media: {df_gelato['temperatura'].mean():.2f}°C, "
      f"Mediana: {df_gelato['temperatura'].median():.2f}°C, "
      f"Desvio Padrao: {df_gelato['temperatura'].std():.2f}°C")
print(f"Vendas - Media: R$ {df_gelato['vendas'].mean():.2f}, "
      f"Mediana: R$ {df_gelato['vendas'].median():.2f}, "
      f"Desvio Padrao: R$ {df_gelato['vendas'].std():.2f}")

## 4. Analise Bivariada

Relacao entre temperatura e vendas, incluindo grafico de dispersao e mapa de calor de correlacao.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Scatter plot: Temperatura vs Vendas
sns.scatterplot(
    data=df_gelato, x="temperatura", y="vendas",
    alpha=0.6, color="#FF9800", edgecolor="white", s=60, ax=axes[0]
)

# Linha de tendencia
z = np.polyfit(df_gelato["temperatura"], df_gelato["vendas"], 1)
p = np.poly1d(z)
temp_range = np.linspace(
    df_gelato["temperatura"].min(),
    df_gelato["temperatura"].max(),
    100
)
axes[0].plot(
    temp_range, p(temp_range), "--", color="#E53935",
    linewidth=2, label=f"Tendencia (y = {z[0]:.1f}x + {z[1]:.1f})"
)
axes[0].legend(fontsize=11)
axes[0].set_title("Temperatura vs Vendas", fontsize=16, fontweight="bold")
axes[0].set_xlabel("Temperatura (°C)", fontsize=12)
axes[0].set_ylabel("Vendas (R$)", fontsize=12)

# Correlacao entre temperatura e vendas
corr_temp_vendas = df_gelato["temperatura"].corr(df_gelato["vendas"])
axes[0].annotate(
    f"r = {corr_temp_vendas:.4f}",
    xy=(0.05, 0.95), xycoords="axes fraction",
    fontsize=14, fontweight="bold",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", edgecolor="gray")
)

# Heatmap de correlacao
numeric_df = df_gelato.select_dtypes(include=[np.number])
corr = numeric_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, annot=True, fmt=".2f", cmap="coolwarm",
    center=0, square=True, linewidths=0.5,
    mask=mask, ax=axes[1],
    cbar_kws={"shrink": 0.8}
)
axes[1].set_title("Mapa de Calor - Correlacao", fontsize=16, fontweight="bold")

plt.suptitle("Analise Bivariada", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print(f"\nCorrelacao Temperatura x Vendas: {corr_temp_vendas:.4f}")
print("Interpretacao: Correlacao forte e positiva - quanto maior a temperatura, maiores as vendas.")

## 5. Analise Temporal

Analise das vendas por dia da semana e por estacao do ano.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- Vendas por dia da semana ---
dias_nomes = {
    0: "Segunda", 1: "Terca", 2: "Quarta",
    3: "Quinta", 4: "Sexta", 5: "Sabado", 6: "Domingo"
}
df_temp = df_gelato.copy()
df_temp["dia_nome"] = df_temp["dia_da_semana"].map(dias_nomes)
ordem_dias = ["Segunda", "Terca", "Quarta", "Quinta", "Sexta", "Sabado", "Domingo"]

sns.barplot(
    data=df_temp, x="dia_nome", y="vendas",
    order=ordem_dias, palette="viridis",
    errorbar="sd", ax=axes[0]
)
axes[0].set_title("Vendas Medias por Dia da Semana", fontsize=16, fontweight="bold")
axes[0].set_xlabel("Dia da Semana", fontsize=12)
axes[0].set_ylabel("Vendas (R$)", fontsize=12)
axes[0].tick_params(axis="x", rotation=45)

# --- Vendas por estacao ---
ordem_estacoes = ["verao", "outono", "inverno", "primavera"]
estacoes_presentes = [e for e in ordem_estacoes if e in df_gelato["estacao"].unique()]

sns.boxplot(
    data=df_gelato, x="estacao", y="vendas",
    order=estacoes_presentes, palette="Set2", ax=axes[1]
)
axes[1].set_title("Distribuicao de Vendas por Estacao", fontsize=16, fontweight="bold")
axes[1].set_xlabel("Estacao do Ano", fontsize=12)
axes[1].set_ylabel("Vendas (R$)", fontsize=12)

plt.suptitle("Analise Temporal", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Estatisticas por dia da semana
print("\n--- Vendas Medias por Dia da Semana ---")
vendas_dia = df_temp.groupby("dia_nome")["vendas"].agg(["mean", "std", "count"])
vendas_dia = vendas_dia.reindex(ordem_dias)
vendas_dia.columns = ["Media (R$)", "Desvio Padrao", "Contagem"]
display(vendas_dia)

# Estatisticas por estacao
print("\n--- Vendas por Estacao ---")
vendas_estacao = df_gelato.groupby("estacao")["vendas"].agg(["mean", "median", "std", "count"])
vendas_estacao = vendas_estacao.reindex(estacoes_presentes)
vendas_estacao.columns = ["Media (R$)", "Mediana (R$)", "Desvio Padrao", "Contagem"]
display(vendas_estacao)

## 6. Correlacoes e Features

Analise do impacto de feriados e finais de semana nas vendas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Efeito de feriados nas vendas ---
df_gelato["feriado_label"] = df_gelato["eh_feriado"].map({0: "Dia Normal", 1: "Feriado"})

sns.boxplot(
    data=df_gelato, x="feriado_label", y="vendas",
    palette=["#64B5F6", "#FF7043"], ax=axes[0]
)
axes[0].set_title("Efeito de Feriados nas Vendas", fontsize=16, fontweight="bold")
axes[0].set_xlabel("Tipo de Dia", fontsize=12)
axes[0].set_ylabel("Vendas (R$)", fontsize=12)

# --- Efeito de finais de semana ---
df_gelato["eh_fim_de_semana"] = df_gelato["dia_da_semana"].isin([5, 6]).astype(int)
df_gelato["fds_label"] = df_gelato["eh_fim_de_semana"].map(
    {0: "Dia Util", 1: "Fim de Semana"}
)

sns.boxplot(
    data=df_gelato, x="fds_label", y="vendas",
    palette=["#81C784", "#FFB74D"], ax=axes[1]
)
axes[1].set_title("Efeito de Fim de Semana nas Vendas", fontsize=16, fontweight="bold")
axes[1].set_xlabel("Tipo de Dia", fontsize=12)
axes[1].set_ylabel("Vendas (R$)", fontsize=12)

plt.suptitle("Analise de Features Categoricas", fontsize=18, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Estatisticas de feriados
print("\n--- Impacto de Feriados ---")
vendas_feriado = df_gelato.groupby("feriado_label")["vendas"].agg(["mean", "median", "std", "count"])
vendas_feriado.columns = ["Media (R$)", "Mediana (R$)", "Desvio Padrao", "Contagem"]
display(vendas_feriado)

media_normal = df_gelato[df_gelato["eh_feriado"] == 0]["vendas"].mean()
media_feriado = df_gelato[df_gelato["eh_feriado"] == 1]["vendas"].mean()
premium_feriado = ((media_feriado - media_normal) / media_normal) * 100
print(f"\nPremium de feriado: {premium_feriado:+.1f}%")

# Estatisticas de fim de semana
print("\n--- Impacto de Fim de Semana ---")
vendas_fds = df_gelato.groupby("fds_label")["vendas"].agg(["mean", "median", "std", "count"])
vendas_fds.columns = ["Media (R$)", "Mediana (R$)", "Desvio Padrao", "Contagem"]
display(vendas_fds)

media_util = df_gelato[df_gelato["eh_fim_de_semana"] == 0]["vendas"].mean()
media_fds = df_gelato[df_gelato["eh_fim_de_semana"] == 1]["vendas"].mean()
premium_fds = ((media_fds - media_util) / media_util) * 100
print(f"\nPremium de fim de semana: {premium_fds:+.1f}%")

# Limpeza das colunas auxiliares
df_gelato.drop(columns=["feriado_label", "eh_fim_de_semana", "fds_label"], inplace=True)

## 7. Insights Principais

Com base na analise exploratoria realizada, os principais insights identificados sao:

1. **Correlacao forte entre temperatura e vendas (~0.89):** A temperatura e o fator mais determinante para as vendas de sorvete. A relacao e linear e positiva, indicando que dias mais quentes geram significativamente mais receita.

2. **Efeito de fim de semana:** As vendas tendem a ser maiores nos finais de semana (sabado e domingo), quando ha maior fluxo de clientes e consumo de lazer.

3. **Premium de feriados:** Dias de feriado apresentam um acrescimo nas vendas em relacao a dias normais, reforçando a importancia de considerar esta variavel no modelo.

4. **Padroes sazonais (verao > inverno):** As vendas no verao sao substancialmente superiores as do inverno, com primavera e outono apresentando valores intermediarios. A sazonalidade esta diretamente ligada a variacao de temperatura.

5. **Recomendacao para modelagem:** Dado o comportamento predominantemente linear da relacao temperatura-vendas, recomenda-se iniciar com modelos de regressao linear, seguidos por modelos mais complexos (Random Forest, Gradient Boosting) para capturar interacoes entre features como dia da semana, feriados e estacao.

## 8. Conclusao

A analise exploratoria dos dados do projeto **Gelato Magico** revelou um dataset de boa qualidade, com poucas ou nenhumas inconsistencias e valores ausentes. As principais conclusoes sao:

- **Qualidade dos dados:** O dataset esta limpo e bem estruturado, com tipos de dados adequados e sem valores ausentes significativos.
- **Variavel alvo (vendas):** Apresenta distribuicao aproximadamente normal, facilitando a modelagem com algoritmos de regressao.
- **Feature principal (temperatura):** Forte poder preditivo, com correlacao linear elevada com as vendas.
- **Features complementares:** Dia da semana, feriados e estacao do ano adicionam informacao relevante para melhorar a precisao do modelo.
- **Prontidao para modelagem:** Os dados estao prontos para as etapas de feature engineering e treinamento de modelos de Machine Learning.

**Proximo passo:** Aplicar engenharia de features e treinar modelos preditivos utilizando Azure Machine Learning (DP-100).